# Numerical failure and direct loss--link selection

This notebook reproduces the direct numerical-failure comparison that preceded the reference-based selection experiment. It uses a correctly specified treatment-specific dictionary. The selection criterion is the observed held-out imbalance squared plus the estimated variance. This criterion is a diagnostic benchmark, not the reference-based procedure implemented in notebook 09.


Each replication draws two independent samples. The diagnostic sample determines the selected loss, and the evaluation sample supplies the reported squared error. Selecting and evaluating on the same fits would reuse the same noise and understate the selected error.

In [ ]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
for candidate in (REPO_ROOT, *REPO_ROOT.parents):
    if (candidate / "src" / "genriesz").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from inside the genriesz repository.")

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from genriesz.experiments import (
    COMPATIBLE_LOSSES,
    ESTIMANDS,
    ESTIMATORS_ALL,
    RANDOM_SEED,
    TREATMENT_INDEX,
    SelectedColumnsBasis,
    fit_grr_love_plot_data,
    fit_matching_ate,
    fit_one_grr,
    fit_one_grr_with_basis,
    fit_one_incompatible,
    fit_one_plugin_logistic,
    load_ihdp_replication,
    load_lalonde,
    make_basis,
    make_dimension_data,
    make_kang_schafer_data,
    make_kernel_gp_data,
    make_score_guided_data,
    make_simulation_data,
    result_to_rows,
    summarize_estimates,
    true_theta,
)

DATA_DIR = REPO_ROOT / "notebooks" / "experiments" / "data"

TABLE_CONFIG = {"float_format": "{:.4f}", "max_rows": 200}
PLOT_CONFIG = {
    "figure_size": (9.0, 5.2),
    "figure_size_wide": (12.0, 5.2),
    "title_fontsize": 14,
    "axis_fontsize": 12,
    "tick_fontsize": 10,
    "legend_fontsize": 10,
    "line_width": 2.0,
    "marker_size": 5,
    "box_width": 0.70,
    "grid_alpha": 0.30,
    "dpi": 140,
    "squared_error_y_scale": "log",
    "squared_error_floor": 1e-12,
}
METHOD_COLORS = {
    "SQ": "#4C78A8",
    "UKL": "#F58518",
    "BKL": "#54A24B",
    "BP(0.5)": "#B279A2",
    "rkhs": "#4C78A8",
    "polynomial": "#F58518",
    "rf": "#54A24B",
    "rff": "#E45756",
    "matching": "#72B7B2",
}
DISPLAY_LABELS = {
    "ra": "RA",
    "rw": "RW",
    "arw": "ARW",
    "tmle": "TMLE",
    "rkhs": "RKHS",
    "polynomial": "Polynomial",
    "rf": "Random Forest",
    "rff": "Random Fourier Features",
    "matching": "Nearest-Neighbor Matching",
    "regressor": "Regressor",
    "covariate": "Covariate",
}
_LABEL_COLUMNS = ("estimator", "basis", "basis_mode", "loss", "loss_link_pair")


def label_of(value):
    return DISPLAY_LABELS.get(str(value), str(value))


def prettify_method(text):
    out = str(text)
    for key, value in DISPLAY_LABELS.items():
        out = re.sub(r"(?<![A-Za-z0-9_])" + re.escape(key) + r"(?![A-Za-z0-9_])", value, out)
    return out


def display_table(df: pd.DataFrame, *, caption: str | None = None, digits: int = 4):
    table = df.copy()
    for column in _LABEL_COLUMNS:
        if column in table.columns:
            table[column] = table[column].map(label_of)
    numeric = table.select_dtypes(include=[np.number]).columns
    table[numeric] = table[numeric].round(digits)
    styler = table.style.format(precision=digits)
    if caption is not None:
        styler = styler.set_caption(caption)
    display(styler)

pd.options.display.max_rows = TABLE_CONFIG["max_rows"]

from genriesz.experiments import CoverageDiagnosticBasis, make_coverage_diagnostic_data


In [ ]:
N_REPLICATIONS = 200
SAMPLE_SIZE = 2000
OVERLAP_SETTINGS = {"Strong": 0.5, "Weak": 2.5}
RIEZ_LAMBDA = 1e-2
FOLDS = 2


In [ ]:
candidate_rows = []
selection_rows = []
for overlap_name, overlap_scale in OVERLAP_SETTINGS.items():
    for replication in range(N_REPLICATIONS):
        base_seed = 2_000_000 + 10_007 * replication + int(100 * overlap_scale)
        rows = {}
        for sample_role, seed_offset in (("diagnostic", 0), ("evaluation", 5_009)):
            data = make_coverage_diagnostic_data(
                n=SAMPLE_SIZE,
                seed=base_seed + seed_offset,
                overlap_scale=overlap_scale,
            )
            role_rows = []
            for loss_spec in COMPATIBLE_LOSSES:
                fit_rows = fit_one_grr_with_basis(
                    data,
                    estimand="ATE",
                    loss_spec=loss_spec,
                    representer_basis=CoverageDiagnosticBasis(include_quadratic=True),
                    cross_fit=True,
                    lam=RIEZ_LAMBDA,
                    folds=FOLDS,
                    estimators=("arw",),
                    random_state=replication,
                    label_info={
                        "overlap": overlap_name,
                        "replication": replication,
                        "sample_role": sample_role,
                    },
                )
                role_rows.extend(fit_rows)
            rows[sample_role] = pd.DataFrame(role_rows)
            candidate_rows.extend(role_rows)
        diagnostic = rows["diagnostic"]
        evaluation = rows["evaluation"]
        stable = diagnostic[diagnostic["status"] == "ok"].copy()
        if stable.empty:
            selection_rows.append({
                "overlap": overlap_name,
                "replication": replication,
                "status": "no_admissible_candidate",
            })
            continue
        stable["criterion"] = stable["held_out_imbalance_max"] ** 2 + stable["se"] ** 2
        selected_loss = stable.loc[stable["criterion"].idxmin(), "loss"]
        eval_ok = evaluation[evaluation["status"] == "ok"]
        selected_eval = eval_ok[eval_ok["loss"] == selected_loss]
        if selected_eval.empty:
            selection_rows.append({
                "overlap": overlap_name,
                "replication": replication,
                "status": "selected_candidate_failed_on_evaluation",
                "selected_loss": selected_loss,
            })
            continue
        oracle = eval_ok.loc[eval_ok["squared_error"].idxmin()]
        selection_rows.append({
            "overlap": overlap_name,
            "replication": replication,
            "status": "ok",
            "selected_loss": selected_loss,
            "selected_squared_error": float(selected_eval["squared_error"].iloc[0]),
            "oracle_squared_error": float(oracle["squared_error"]),
        })
candidate_results = pd.DataFrame(candidate_rows)
selection_results = pd.DataFrame(selection_rows)

In [ ]:
evaluation_candidates = candidate_results[
    candidate_results["sample_role"] == "evaluation"
]
failure_table = (
    evaluation_candidates.assign(failed=evaluation_candidates["status"] != "ok")
    .groupby(["overlap", "loss"], as_index=False)["failed"]
    .agg(failures="sum", failure_rate="mean")
)
selection_summary = (
    selection_results[selection_results["status"] == "ok"]
    .groupby("overlap", as_index=False)
    .agg(
        selected_rmse=("selected_squared_error", lambda x: float(np.sqrt(np.mean(x)))),
        oracle_rmse=("oracle_squared_error", lambda x: float(np.sqrt(np.mean(x)))),
        stable_replications=("replication", "count"),
    )
)
display_table(failure_table, caption="Numerical failures by loss and overlap setting")
display_table(selection_summary, caption="Direct held-out-imbalance selection and the infeasible oracle")


In [ ]:
selection_frequency = (
    selection_results[selection_results["status"] == "ok"]
    .groupby(["overlap", "selected_loss"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)
selection_frequency["frequency"] = selection_frequency.groupby("overlap")["count"].transform(
    lambda values: values / values.sum()
)
for overlap_name in OVERLAP_SETTINGS:
    panel = selection_frequency[selection_frequency["overlap"] == overlap_name].copy()
    figure, axis = plt.subplots(figsize=(7.0, 4.5), dpi=PLOT_CONFIG["dpi"])
    axis.bar(panel["selected_loss"], panel["frequency"])
    axis.set_ylim(0.0, 1.0)
    axis.set_xlabel("Selected loss", fontsize=PLOT_CONFIG["axis_fontsize"])
    axis.set_ylabel("Selection frequency", fontsize=PLOT_CONFIG["axis_fontsize"])
    axis.set_title(f"{overlap_name} overlap", fontsize=PLOT_CONFIG["title_fontsize"])
    axis.grid(axis="y", alpha=PLOT_CONFIG["grid_alpha"])
    figure.tight_layout()
    plt.show()
